# DESCN: Deep Entire Space Cross Networks for Individual Treatment Effect Estimation


## 1. Causal Inference


### 1.1 Potential Outcomes

For a unit $i$ with covariates $x_i$ and binary treatment $w_i \in \{0,1\}$:

- $Y_i(1)$: potential outcome under treatment
- $Y_i(0)$: potential outcome under control
- Observed outcome: $y_i = w_iY_i(1)+(1-w_i)Y_i(0)$

Only one potential outcome is observed for each unit; the other is counterfactual.

### 1.2 Estimands and Functions

| Quantity | Definition |
|---|---|
| ITE / CATE | $\tau(x)=\mathbb{E}[Y(1)-Y(0)\mid X=x]$ |
| ATE | $\mathbb{E}[Y(1)-Y(0)]$ |
| ATT | $\mathbb{E}[Y(1)-Y(0)\mid W=1]$ |
| Treated response | $\mu_1(x)=\mathbb{E}[Y\mid W=1,X=x]$ |
| Control response | $\mu_0(x)=\mathbb{E}[Y\mid W=0,X=x]$ |
| Propensity | $\pi(x)=P(W=1\mid X=x)$ |

The predicted individual treatment effect is $\hat{\tau}(x)=\hat{\mu}_1(x)-\hat{\mu}_0(x)$.

### 1.3 Assumptions

1. **Consistency**: $y_i=Y_i(w_i)$.
2. **Ignorability**: $Y(1),Y(0)\perp\!\!\!\perp W\mid X$.
3. **Overlap**: $0<\pi(x)<1$.

### 1.4 Modeling Issues

- **Treatment bias**: treated and control feature distributions differ.
- **Sample imbalance**: treated and control group sizes differ.

---


## 2. DESCN Architecture

This section lists the equations used by the implementation. The notebook uses one shared architecture; each model name corresponds to a different loss-weight configuration.

### 2.1 Entire Space Network (ESN)

ESN models treatment and response jointly:

$$\text{ESTR}=P(Y,W=1\mid X)=P(Y\mid W=1,X)P(W=1\mid X)=\mu_1\pi$$
$$\text{ESCR}=P(Y,W=0\mid X)=P(Y\mid W=0,X)P(W=0\mid X)=\mu_0(1-\pi)$$

The labels for the three ESN losses are $w_i$, $y_iw_i$, and $y_i(1-w_i)$:

$$L_{\pi}=\frac{1}{n}\sum_i l(w_i,\hat{\pi}(x_i))$$
$$L_{ESTR}=\frac{1}{n}\sum_i l(y_iw_i,\hat{\mu}_1(x_i)\hat{\pi}(x_i))$$
$$L_{ESCR}=\frac{1}{n}\sum_i l(y_i(1-w_i),\hat{\mu}_0(x_i)(1-\hat{\pi}(x_i)))$$
$$L_{ESN}=\alpha L_{\pi}+\beta_1L_{ESTR}+\beta_0L_{ESCR}$$

The same structure is related to inverse-probability weighting:

$$ATE=\mathbb{E}\left[\frac{WY}{\pi(X)}\right]-\mathbb{E}\left[\frac{(1-W)Y}{1-\pi(X)}\right]$$

### 2.2 X-network

X-network adds a pseudo treatment-effect logit $s_{\tau}(x)$ and defines cross responses in logit space:

$$\mu'_1=\sigma\left(\sigma^{-1}(\mu_0)+s_{\tau}(x)\right)$$
$$\mu'_0=\sigma\left(\sigma^{-1}(\mu_1)-s_{\tau}(x)\right)$$

The losses are:

$$L_{TR}=\frac{1}{|T|}\sum_{i\in T}l(y_i,\hat{\mu}_1(x_i)),\quad L_{CR}=\frac{1}{|C|}\sum_{i\in C}l(y_i,\hat{\mu}_0(x_i))$$
$$L_{CrossTR}=\frac{1}{|T|}\sum_{i\in T}l(y_i,\hat{\mu}'_1(x_i)),\quad L_{CrossCR}=\frac{1}{|C|}\sum_{i\in C}l(y_i,\hat{\mu}'_0(x_i))$$

### 2.3 Loss-Weight Configurations

The table below is not a results table. It shows which loss terms are active for each model. These values mirror `DESCN/conf4models/ACIC2019/*.txt` because the notebook defaults to ACIC.

| ACIC model | $\alpha$ | $\beta_1,\beta_0$ | $\gamma_1,\gamma_0$ | IPM $\lambda$ | TR/CR |
|---|---:|---:|---:|---:|---:|
| TARNet | 0 | 0, 0 | 0, 0 | 0 | 1, 1 |
| CFR(MMD) | 0 | 0, 0 | 0, 0 | 2 | 1, 2 |
| X-network | 0 | 0, 0 | 0.8, 0.3 | 0 | 1, 2 |
| DESCN | 0.5 | 1, 1 | 0.8, 0.3 | 0 | 0, 0 |

For Lazada production, the code switches to `DESCN/conf4models/lzd_real_data/*.txt` values.

### 2.4 DESCN Loss

DESCN combines ESN and X-network losses:

$$L_{DESCN}=L_{ESN}+\gamma_1L_{CrossTR}+\gamma_0L_{CrossCR}$$
$$=\alpha L_{\pi}+\beta_1L_{ESTR}+\beta_0L_{ESCR}+\gamma_1L_{CrossTR}+\gamma_0L_{CrossCR}$$

For DESCN, direct TR/CR losses are disabled; TR and CR are learned through ESTR/ESCR and cross losses.

---


## 3. Implementation


### 3.1 Setup


In [6]:
import gc, os, random, warnings

import numpy as np
import tensorflow as tf
from tqdm import tqdm

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # suppress TF C++ INFO/WARN logs

warnings.filterwarnings('ignore', category=FutureWarning)
tf.get_logger().setLevel('ERROR')

# ---- Reproducibility ----
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(2)

# ---- Runtime / CUDA visibility ----
gpu_devices = tf.config.list_physical_devices('GPU')
for gpu_device in gpu_devices:
    try:
        tf.config.experimental.set_memory_growth(gpu_device, True)
    except RuntimeError as error:
        print(f'Could not set memory growth for {gpu_device.name}: {error}')

build_info = tf.sysconfig.get_build_info()
print(f'TF {tf.__version__}  |  GPUs visible: {len(gpu_devices)}')
if gpu_devices:
    for gpu_index, gpu_device in enumerate(gpu_devices):
        try:
            gpu_details = tf.config.experimental.get_device_details(gpu_device)
        except Exception:
            gpu_details = {}
        gpu_name = gpu_details.get('device_name', gpu_device.name)
        print(f'  GPU {gpu_index}: {gpu_name}')
    print(f"  CUDA build: {build_info.get('cuda_version', 'unknown')}")
    print(f"  cuDNN build: {build_info.get('cudnn_version', 'unknown')}")
else:
    print('  Running on CPU. On RunPod, check nvidia-smi and tensorflow[and-cuda] install if this is unexpected.')


TF 2.17.1  |  GPUs visible: 1
  GPU 0: METAL
  CUDA build: unknown
  cuDNN build: unknown


### 3.2 Data

#### Lazada Production Dataset

The paper uses a biased production train set and a randomized test set:

- train: approximately 927K public samples, 83 covariates
- test: approximately 182K public randomized samples
- binary treatment and binary outcome

The Lazada cells are kept for switching `DATASET='lazada'`; the current notebook default is ACIC.

In [7]:
# import zipfile, urllib.request
# from pathlib import Path

# DATA_PATH = Path('./data')
# DATA_PATH.mkdir(exist_ok=True)

# lazada_zip = DATA_PATH / 'lzd_data_public.zip'
# lazada_dir = DATA_PATH / 'lzd_data_public'

# if not lazada_dir.exists():
#     url = 'https://www.dropbox.com/s/07r7592h9mfijsb/lzd_data_public.zip?dl=1'
#     print(f'Downloading Lazada dataset ({url})...')
#     urllib.request.urlretrieve(url, lazada_zip)
    
#     print('Extracting...')
#     with zipfile.ZipFile(lazada_zip, 'r') as f:
#         f.extractall(DATA_PATH)
#     lazada_zip.unlink()
#     print('Done.')
# else:
#     print(f'Lazada dataset already exists at {lazada_dir}')


In [8]:
# def load_lazada_data():
#     import pandas as pd
    
#     train_df = pd.read_csv(lazada_dir / 'full_trainset.csv')
#     test_df = pd.read_csv(lazada_dir / 'full_testset.csv')
    
#     # 83 features: f0–f82
#     feature_cols = [c for c in train_df.columns if c.startswith('f')]
    
#     def df_to_dict(df):
#         return {
#             'x': df[feature_cols].values.astype(np.float32),
#             't': df['is_treat'].values.astype(np.float32),
#             'yf': df['label'].values.astype(np.float32),
#             'ycf': None,        # no counterfactual in real data
#             'mu0': None,
#             'mu1': None,
#             'tau': None,        # no ground truth ITE
#             'e': np.zeros(len(df), dtype=np.float32),  # all 0 = biased (train) / RCT handled below
#         }
    
#     train_data = df_to_dict(train_df)
#     test_data = df_to_dict(test_df)
#     test_data['e'] = np.ones(len(test_df), dtype=np.float32)  # test is RCT
    
#     print(f'Lazada train: {train_data["x"].shape[0]:,} samples, {train_data["x"].shape[1]} features')
#     print(f'  Treated: {train_data["t"].sum():.0f} ({train_data["t"].mean()*100:.1f}%)')
#     print(f'  Outcome rate (treated): {train_data["yf"][train_data["t"]==1].mean():.3f}')
#     print(f'  Outcome rate (control): {train_data["yf"][train_data["t"]==0].mean():.3f}')
#     print(f'Lazada test (RCT): {test_data["x"].shape[0]:,} samples')
#     print(f'  Treated: {test_data["t"].sum():.0f} ({test_data["t"].mean()*100:.1f}%)')
#     print(f'  Outcome rate (treated): {test_data["yf"][test_data["t"]==1].mean():.3f}')
#     print(f'  Outcome rate (control): {test_data["yf"][test_data["t"]==0].mean():.3f}')
    
#     return train_data, test_data

# print(f"Loaded Lazada dataset from {lazada_dir}")

#### ACIC 2019 Epilepsy (Mod 1-4)

Default dataset for this notebook. Generates all 4 Epilepsy DGPs (Mod 1-4) and stacks them into 20 experiments.

The ACIC data is generated from `SG_Generate_High_Dim_Binary/generate_simEpilepsy.R` (all 4 Mods) using the UCI Epileptic Seizure Recognition covariates. The notebook expects DESCN-format files:

- `syn_bin_set.5.train.npz`
- `syn_bin_set.5.test.npz`

`RBL_FINAL/create_acic_npz.py` generates all 4 Mods and records the generation summary.

In [9]:
# ===== Lazada Production (real data) =====
# training_data, test_data = load_lazada_data()

# ===== ACIC 2019 Epilepsy Mod 4 =====
# Official source: DESCN/data/ACIC2019_epilepsy_dataset/ACIC_2019_Generate_DGPs.zip
# R script: ACIC_2019_Generate_DGPs/SG_Generate_High_Dim_Binary/generate_simEpilepsy.R
# Requires the UCI epilepsy `data.csv` next to the R script before running R.
from pathlib import Path
import subprocess
import sys
from scipy.special import expit as plogis

# psi0 for Mod 4 (paper reports 0.2916274). Aggregate ATE across all 4 Mods is ~0.23.
ACIC_EXPECTED_POPULATION_ATE_MOD4 = 0.2916274
# Resolve repo root whether CWD is RBL_FINAL/ (VS Code) or repo root (CLI).
_ACIC_ROOT = Path.cwd()
if not (_ACIC_ROOT / "DESCN").exists():
    _ACIC_ROOT = _ACIC_ROOT.parent
ACIC_DATA_DIR = _ACIC_ROOT / "DESCN" / "data" / "ACIC2019_epilepsy_dataset"
ACIC_DGP_ZIP = ACIC_DATA_DIR / "ACIC_2019_Generate_DGPs.zip"
ACIC_TRAIN_NPZ = ACIC_DATA_DIR / "syn_bin_set.5.train.npz"
ACIC_TEST_NPZ = ACIC_DATA_DIR / "syn_bin_set.5.test.npz"

# Keep this True for a runnable demo when the official .npz files are absent.
# Set to False if you want the notebook to fail fast until the R-generated ACIC
# files have been converted to the DESCN .npz format.
ALLOW_ACIC_SURROGATE_FALLBACK = False
SURROGATE_SAMPLES_PER_EXPERIMENT = 4000


def load_descn_npz(path):
    """Load the public DESCN .npz shape: x=(n, d, exp), labels=(n, exp)."""
    data_in = np.load(path, allow_pickle=True)
    data = {
        "x": data_in["x"].astype(np.float32),
        "t": data_in["t"].astype(np.float32),
        "yf": data_in["yf"].astype(np.float32),
        "e": data_in["e"].astype(np.float32) if "e" in data_in else np.zeros_like(data_in["yf"], dtype=np.float32),
        "ycf": data_in["ycf"].astype(np.float32) if "ycf" in data_in else None,
        "mu0": data_in["mu0"].astype(np.float32) if "mu0" in data_in else None,
        "mu1": data_in["mu1"].astype(np.float32) if "mu1" in data_in else None,
        "tau": data_in["tau"].astype(np.float32) if "tau" in data_in else None,
        "source": str(path),
        "is_official_acic": True,
    }
    return data


def make_temp_mod2(d, keep_coef):
    random_cols = np.array([1, 51, 81, 111, 151]) - 1
    coef_idx = keep_coef[:-1]
    w_ratio = np.log(np.abs(d[:, coef_idx + 1]) + 1) / (np.abs(d[:, coef_idx + 3]) + 1)
    w_interact = d[:, random_cols] * d[:, random_cols + 10]
    return np.column_stack([d[:, random_cols], d[:, random_cols + 10], w_ratio, w_interact])


def generate_acic_mod4_surrogate(
    experiment_count=5,
    samples_per_experiment=SURROGATE_SAMPLES_PER_EXPERIMENT,
    pop_seed=42,
    bootstrap_seed=4,
):
    """Python smoke-data fallback, not the paper dataset.

    The formulas mirror Mod 4, but the covariate population is synthetic because
    the official R flow requires UCI epilepsy `data.csv`. Use only to verify
    that the model/loss code executes.
    """
    n_pop, n_features = 11500, 178
    rng = np.random.RandomState(pop_seed)
    latent = rng.normal(0, 1, (n_pop, 30))
    projection = rng.normal(0, 1, (30, n_features))
    d = latent @ projection + 0.3 * rng.normal(0, 1, (n_pop, n_features))
    d = (d - d.mean(axis=0)) / d.std(axis=0)

    keep_coef = np.arange(0, n_features, 4)
    temp_mod2 = make_temp_mod2(d, keep_coef)

    rng40 = np.random.RandomState(40)
    beta_a = rng40.uniform(-0.1, 0.12, temp_mod2.shape[1]) / temp_mod2.std(axis=0)
    logit_a = -0.1 + temp_mod2 @ beta_a

    beta_y = 2 * beta_a
    beta_y[:5] = 0
    logit_drs = -1.8 + temp_mod2 @ beta_y + d[:, 149] * (-0.005) + d[:, 159] * (-0.02)
    population_ate = float(np.mean(plogis(2 + 0.01 * d[:, 159] + logit_drs) - plogis(logit_drs)))

    rng4 = np.random.RandomState(bootstrap_seed)

    def draw_experiment():
        sample_idx = rng4.choice(n_pop, samples_per_experiment, replace=True)
        treatment = rng4.binomial(1, plogis(logit_a[sample_idx])).astype(np.float32)
        y_logit = treatment * (2 + 0.01 * d[sample_idx, 159]) + logit_drs[sample_idx]
        outcome = rng4.binomial(1, plogis(y_logit)).astype(np.float32)
        mu0 = plogis(logit_drs[sample_idx]).astype(np.float32)
        mu1 = plogis(2 + 0.01 * d[sample_idx, 159] + logit_drs[sample_idx]).astype(np.float32)
        tau = (mu1 - mu0).astype(np.float32)
        ycf = ((1 - treatment) * mu1 + treatment * mu0).astype(np.float32)
        return d[sample_idx].astype(np.float32), treatment, outcome, ycf, mu0, mu1, tau

    train_draws = [draw_experiment() for _ in range(experiment_count)]
    test_draws = [draw_experiment() for _ in range(experiment_count)]

    def stack(draws):
        return {
            "x": np.stack([draw[0] for draw in draws], axis=2),
            "t": np.stack([draw[1] for draw in draws], axis=1),
            "yf": np.stack([draw[2] for draw in draws], axis=1),
            "ycf": np.stack([draw[3] for draw in draws], axis=1),
            "mu0": np.stack([draw[4] for draw in draws], axis=1),
            "mu1": np.stack([draw[5] for draw in draws], axis=1),
            "tau": np.stack([draw[6] for draw in draws], axis=1),
            "e": np.zeros((samples_per_experiment, experiment_count), dtype=np.float32),
            "source": "python_surrogate_not_official_acic",
            "is_official_acic": False,
            "surrogate_population_ate": population_ate,
        }

    return stack(train_draws), stack(test_draws)


def ensure_acic_npz():
    """Generate ignored ACIC .npz files on a fresh server checkout."""
    if ACIC_TRAIN_NPZ.exists() and ACIC_TEST_NPZ.exists():
        return

    # Notebook CWD may be RBL_FINAL/ (VS Code) or repo root (CLI).
    generator_script = Path("create_acic_npz.py")
    if not generator_script.exists():
        generator_script = Path("RBL_FINAL/create_acic_npz.py")
    if not generator_script.exists():
        raise FileNotFoundError(
            f"Missing {generator_script}. Commit this script or run it manually before loading ACIC (all 4 Mods)."
        )

    print("ACIC .npz files not found; generating them with RBL_FINAL/create_acic_npz.py (4 Mods) ...")
    try:
        subprocess.run([sys.executable, str(generator_script)], check=True)
    except subprocess.CalledProcessError as error:
        raise RuntimeError(
            "Failed to generate ACIC .npz files. Exact ACIC generation requires R/Rscript on the server."
        ) from error


def load_acic_data():
    ensure_acic_npz()
    if ACIC_TRAIN_NPZ.exists() and ACIC_TEST_NPZ.exists():
        print(f"Loading official DESCN ACIC npz files:\n  {ACIC_TRAIN_NPZ}\n  {ACIC_TEST_NPZ}")
        return load_descn_npz(ACIC_TRAIN_NPZ), load_descn_npz(ACIC_TEST_NPZ)

    message = (
        "Official DESCN ACIC .npz files were not found. The paper ACIC data is generated by R from "
        f"{ACIC_DGP_ZIP}. Put converted files at:\n"
        f"  {ACIC_TRAIN_NPZ}\n  {ACIC_TEST_NPZ}\n"
        "The R script also needs the UCI epilepsy data.csv in its working directory."
    )
    if not ALLOW_ACIC_SURROGATE_FALLBACK:
        raise FileNotFoundError(message)

    print(message)
    print("Falling back to Python surrogate data for smoke testing only.")
    return generate_acic_mod4_surrogate()


training_data, test_data = load_acic_data()
true_ate = (
    ACIC_EXPECTED_POPULATION_ATE_MOD4
    if training_data.get("is_official_acic")
    else training_data.get("surrogate_population_ate")
)

print("\n--- ACIC data status ---")
print(f"Source: {training_data['source']}")
print(f"Official ACIC npz: {training_data['is_official_acic']}")
if training_data["is_official_acic"]:
    print(f"Paper population ATE (Mod 4): {ACIC_EXPECTED_POPULATION_ATE_MOD4:.7f}")
else:
    print(f"Surrogate population ATE: {true_ate:.6f}; paper R DGP ATE (Mod 4): {ACIC_EXPECTED_POPULATION_ATE_MOD4:.7f}")
    print("Do not compare surrogate metrics against KDD Table 2.")

print(f"Train x shape: {training_data['x'].shape} | Test x shape: {test_data['x'].shape}")
print(f"Train treated ratio: {training_data['t'].mean():.3f} | Test treated ratio: {test_data['t'].mean():.3f}")
print(f"Train outcome rate: {training_data['yf'].mean():.3f} | Test outcome rate: {test_data['yf'].mean():.3f}")


Loading official DESCN ACIC npz files:
  /Users/miphu/Projects/DAT301m/DESCN/data/ACIC2019_epilepsy_dataset/syn_bin_set.5.train.npz
  /Users/miphu/Projects/DAT301m/DESCN/data/ACIC2019_epilepsy_dataset/syn_bin_set.5.test.npz

--- ACIC data status ---
Source: /Users/miphu/Projects/DAT301m/DESCN/data/ACIC2019_epilepsy_dataset/syn_bin_set.5.train.npz
Official ACIC npz: True
Paper population ATE (Mod 4): 0.2916274
Train x shape: (40000, 178, 20) | Test x shape: (40000, 178, 20)
Train treated ratio: 0.461 | Test treated ratio: 0.461
Train outcome rate: 0.421 | Test outcome rate: 0.421


In [10]:
# ==============================================================================
# Model Configuration — mirrors conf4models/*.txt from the original DESCN repo
# ==============================================================================
# All models share the same backbone (DESCN class). Only the loss weights differ.
#
# ACIC: synthetic Epilepsy Mod 4 config (from conf4models/ACIC2019/)
# Lazada: Production config (from conf4models/lzd_real_data/)
#
# Set DATASET below to switch.
# ==============================================================================

DATASET = 'acic'  # 'acic' or 'lazada'

if DATASET == 'lazada':
    HYPERPARAMS = {
        'share_dim': 128,
        'base_dim': 64,
        'batch_size': 5000,
        'lr': 0.001,
        'l2': 0.001,
        'dropout': 0.1,
        'normalize': 'divide',
        'use_batchnorm': True,
    }
    TRAIN_CONFIG = {
        'epochs': 5,
        'decay_rate': 0.95,
        'decay_step_size': 1,
        'val_ratio': 0.2,
        'experiment_train_fraction': 0.9,
        'reweight_sample': True,
        'selection_metric': 'auuc',
        'prediction_output_dir': 'results/lzd_real_tf',
    }
    EXPERIMENT_COUNT = 5
    MODEL_CONFIGS = {
        'TARNet': {
            'prpsy_w': 0, 'escvr1_w': 0, 'escvr0_w': 0,
            'h1_w': 1, 'h0_w': 1,
            'mu1hat_w': 0, 'mu0hat_w': 0,
            'imb_dist_w': 0,
        },
        'CFR_MMD': {
            'prpsy_w': 0, 'escvr1_w': 0, 'escvr0_w': 0,
            'h1_w': 1, 'h0_w': 1,
            'mu1hat_w': 0, 'mu0hat_w': 0,
            'imb_dist_w': 0.1, 'imb_dist': 'mmd',
        },
        'X_network': {
            'prpsy_w': 0, 'escvr1_w': 0, 'escvr0_w': 0,
            'h1_w': 2, 'h0_w': 2,
            'mu1hat_w': 2, 'mu0hat_w': 1,
            'imb_dist_w': 0,
        },
        'DESCN': {
            'prpsy_w': 0.5, 'escvr1_w': 0.5, 'escvr0_w': 1.0,
            'h1_w': 0, 'h0_w': 0,
            'mu1hat_w': 1.0, 'mu0hat_w': 0.5,
            'imb_dist_w': 0, 'imb_dist': 'wass',
        },
    }
else:
    HYPERPARAMS = {
        'share_dim': 256,
        'base_dim': 128,
        'batch_size': 500,
        'lr': 0.001,
        'l2': 0.01,
        'dropout': 0.0,
        'normalize': 'divide',
        'use_batchnorm': False,
    }
    TRAIN_CONFIG = {
        'epochs': 15,
        'decay_rate': 1.0,
        'decay_step_size': 1,
        'val_ratio': 0.2,
        'reweight_sample': True,
        'selection_metric': 'sqrt_pehe',
        'prediction_output_dir': 'results/acic_tf',
    }
    # 4 Mods × 5 experiments per Mod = 20 total experiments
EXPERIMENT_COUNT = min(20, training_data['x'].shape[2] if training_data['x'].ndim == 3 else 20)
MODEL_CONFIGS = {
        'TARNet': {
            'prpsy_w': 0, 'escvr1_w': 0, 'escvr0_w': 0,
            'h1_w': 1, 'h0_w': 2,
            'mu1hat_w': 0, 'mu0hat_w': 0,
            'imb_dist_w': 0,
        },
        'CFR_MMD': {
            'prpsy_w': 0, 'escvr1_w': 0, 'escvr0_w': 0,
            'h1_w': 1, 'h0_w': 2,
            'mu1hat_w': 0, 'mu0hat_w': 0,
            'imb_dist_w': 2, 'imb_dist': 'mmd',
        },
        'X_network': {
            'prpsy_w': 0, 'escvr1_w': 0, 'escvr0_w': 0,
            'h1_w': 1, 'h0_w': 2,
            'mu1hat_w': 0.8, 'mu0hat_w': 0.3,
            'imb_dist_w': 0,
        },
        'DESCN': {
            'prpsy_w': 0.5, 'escvr1_w': 1.0, 'escvr0_w': 1.0,
            'h1_w': 0, 'h0_w': 0,
            'mu1hat_w': 0.8, 'mu0hat_w': 0.3,
            'imb_dist_w': 0, 'imb_dist': 'wass',
        },
    }

print(f'Dataset: {DATASET} | Models: {list(MODEL_CONFIGS.keys())}')
print(f'  experiments={EXPERIMENT_COUNT}  selection={TRAIN_CONFIG["selection_metric"]}')
print(f'  share_dim={HYPERPARAMS["share_dim"]}  base_dim={HYPERPARAMS["base_dim"]}  '
      f'epochs={TRAIN_CONFIG["epochs"]}  batch={HYPERPARAMS["batch_size"]}  '
      f'lr={HYPERPARAMS["lr"]}  l2={HYPERPARAMS["l2"]}  dropout={HYPERPARAMS["dropout"]}  '
      f'batchnorm={HYPERPARAMS["use_batchnorm"]}')
print(f'  TARNet:       h1={MODEL_CONFIGS["TARNet"]["h1_w"]} h0={MODEL_CONFIGS["TARNet"]["h0_w"]}')
print(f'  CFR_MMD:      h1={MODEL_CONFIGS["CFR_MMD"]["h1_w"]} h0={MODEL_CONFIGS["CFR_MMD"]["h0_w"]} imb={MODEL_CONFIGS["CFR_MMD"]["imb_dist_w"]}')
print(f'  X_network:    h1={MODEL_CONFIGS["X_network"]["h1_w"]} h0={MODEL_CONFIGS["X_network"]["h0_w"]} xTR={MODEL_CONFIGS["X_network"]["mu1hat_w"]} xCR={MODEL_CONFIGS["X_network"]["mu0hat_w"]}')
print(f'  DESCN:        prpsy={MODEL_CONFIGS["DESCN"]["prpsy_w"]} estr={MODEL_CONFIGS["DESCN"]["escvr1_w"]} escr={MODEL_CONFIGS["DESCN"]["escvr0_w"]} xTR={MODEL_CONFIGS["DESCN"]["mu1hat_w"]} xCR={MODEL_CONFIGS["DESCN"]["mu0hat_w"]}')


Dataset: acic | Models: ['TARNet', 'CFR_MMD', 'X_network', 'DESCN']
  experiments=20  selection=sqrt_pehe
  share_dim=256  base_dim=128  epochs=15  batch=500  lr=0.001  l2=0.01  dropout=0.0  batchnorm=False
  TARNet:       h1=1 h0=2
  CFR_MMD:      h1=1 h0=2 imb=2
  X_network:    h1=1 h0=2 xTR=0.8 xCR=0.3
  DESCN:        prpsy=0.5 estr=1.0 escr=1.0 xTR=0.8 xCR=0.3


### 3.3 Train / Validation / Test Split


In [11]:
from sklearn.model_selection import train_test_split


def to_tensor(array):
    return tf.constant(array, dtype=tf.float32)


def experiment_count_for(data):
    x = data['x']
    return x.shape[2] if x.ndim == 3 else 1


def slice_experiment(data, key, experiment_index):
    value = data.get(key)
    if value is None:
        return None
    if key == 'x':
        return value[:, :, experiment_index] if value.ndim == 3 else value
    return value[:, experiment_index] if value.ndim == 2 else value


def make_tensor_dict(features, outcomes, treatments, randomized_flags, treatment_effect=None):
    return {
        'features': to_tensor(features),
        'outcomes': to_tensor(outcomes.reshape(-1, 1)),
        'treatments': to_tensor(treatments.reshape(-1, 1)),
        'randomized_flags': to_tensor(randomized_flags.reshape(-1, 1)),
        'treatment_effect': to_tensor(treatment_effect.reshape(-1, 1)) if treatment_effect is not None else None,
    }


if DATASET == 'lazada':
    training_features = training_data['x']
    training_outcomes = training_data['yf'].reshape(-1, 1)
    training_treatments = training_data['t'].reshape(-1, 1)
    training_randomized_flags = training_data['e'].reshape(-1, 1)

    sample_count = len(training_features)
    all_training_indices = np.arange(sample_count)
    experiment_splits = []
    validation_random_state = np.random.RandomState(2)
    for experiment_index in range(EXPERIMENT_COUNT):
        _, experiment_indices = train_test_split(
            all_training_indices,
            test_size=TRAIN_CONFIG['experiment_train_fraction'],
            random_state=experiment_index,
            shuffle=True,
        )
        validation_count = int(len(experiment_indices) * TRAIN_CONFIG['val_ratio'])
        train_count = len(experiment_indices) - validation_count
        shuffled_experiment_indices = validation_random_state.permutation(experiment_indices)
        experiment_splits.append({
            'train_indices': shuffled_experiment_indices[:train_count],
            'validation_indices': shuffled_experiment_indices[train_count:],
        })

    test_bundle = make_tensor_dict(
        test_data['x'],
        test_data['yf'],
        test_data['t'],
        test_data['e'],
        test_data.get('tau'),
    )

    def make_experiment_tensors(experiment_index):
        experiment_split = experiment_splits[experiment_index]
        train_indices = experiment_split['train_indices']
        validation_indices = experiment_split['validation_indices']
        return {
            'features_train': to_tensor(training_features[train_indices]),
            'outcomes_train': to_tensor(training_outcomes[train_indices]),
            'treatments_train': to_tensor(training_treatments[train_indices]),
            'randomized_flags_train': to_tensor(training_randomized_flags[train_indices]),
            'features_validation': to_tensor(training_features[validation_indices]),
            'outcomes_validation': to_tensor(training_outcomes[validation_indices]),
            'treatments_validation': to_tensor(training_treatments[validation_indices]),
            'randomized_flags_validation': to_tensor(training_randomized_flags[validation_indices]),
            'features_test': test_bundle['features'],
            'outcomes_test': test_bundle['outcomes'],
            'treatments_test': test_bundle['treatments'],
            'randomized_flags_test': test_bundle['randomized_flags'],
            'treatment_effect_test': test_bundle['treatment_effect'],
        }

    first_split = experiment_splits[0]
    print(f'Prepared {EXPERIMENT_COUNT} Lazada experiment subsets.')
    print(f'  Each: {len(first_split["train_indices"]):,} train / {len(first_split["validation_indices"]):,} val')
    print(f'  Test RCT: {len(test_bundle["features"]):,} samples')
else:
    available_experiments = min(experiment_count_for(training_data), experiment_count_for(test_data))
    EXPERIMENT_COUNT = min(EXPERIMENT_COUNT, available_experiments)

    experiment_splits = []
    validation_random_state = np.random.RandomState(2)
    for experiment_index in range(EXPERIMENT_COUNT):
        outcomes_for_split = slice_experiment(training_data, 'yf', experiment_index)
        sample_count = len(outcomes_for_split)
        validation_count = int(sample_count * TRAIN_CONFIG['val_ratio'])
        shuffled_indices = validation_random_state.permutation(sample_count)
        experiment_splits.append({
            'train_indices': shuffled_indices[:-validation_count],
            'validation_indices': shuffled_indices[-validation_count:],
        })

    def make_experiment_tensors(experiment_index):
        train_features = slice_experiment(training_data, 'x', experiment_index)
        train_outcomes = slice_experiment(training_data, 'yf', experiment_index)
        train_treatments = slice_experiment(training_data, 't', experiment_index)
        train_randomized_flags = slice_experiment(training_data, 'e', experiment_index)
        test_features = slice_experiment(test_data, 'x', experiment_index)
        test_outcomes = slice_experiment(test_data, 'yf', experiment_index)
        test_treatments = slice_experiment(test_data, 't', experiment_index)
        test_randomized_flags = slice_experiment(test_data, 'e', experiment_index)
        test_treatment_effect = slice_experiment(test_data, 'tau', experiment_index)

        experiment_split = experiment_splits[experiment_index]
        train_indices = experiment_split['train_indices']
        validation_indices = experiment_split['validation_indices']
        return {
            'features_train': to_tensor(train_features[train_indices]),
            'outcomes_train': to_tensor(train_outcomes[train_indices].reshape(-1, 1)),
            'treatments_train': to_tensor(train_treatments[train_indices].reshape(-1, 1)),
            'randomized_flags_train': to_tensor(train_randomized_flags[train_indices].reshape(-1, 1)),
            'features_validation': to_tensor(train_features[validation_indices]),
            'outcomes_validation': to_tensor(train_outcomes[validation_indices].reshape(-1, 1)),
            'treatments_validation': to_tensor(train_treatments[validation_indices].reshape(-1, 1)),
            'randomized_flags_validation': to_tensor(train_randomized_flags[validation_indices].reshape(-1, 1)),
            'features_test': to_tensor(test_features),
            'outcomes_test': to_tensor(test_outcomes.reshape(-1, 1)),
            'treatments_test': to_tensor(test_treatments.reshape(-1, 1)),
            'randomized_flags_test': to_tensor(test_randomized_flags.reshape(-1, 1)),
            'treatment_effect_test': to_tensor(test_treatment_effect.reshape(-1, 1)) if test_treatment_effect is not None else None,
        }

    first_data = make_experiment_tensors(0)
    print(f'Prepared {EXPERIMENT_COUNT} ACIC experiments from {"official npz" if training_data["is_official_acic"] else "surrogate data"}.')
    print(f'  Exp 1: {len(first_data["features_train"]):,} train / {len(first_data["features_validation"]):,} val / {len(first_data["features_test"]):,} test')
    print(f'  Treated ratio - Train: {tf.reduce_mean(first_data["treatments_train"]).numpy():.3f} | '
          f'Val: {tf.reduce_mean(first_data["treatments_validation"]).numpy():.3f} | '
          f'Test: {tf.reduce_mean(first_data["treatments_test"]).numpy():.3f}')


Prepared 20 ACIC experiments from official npz.
  Exp 1: 32,000 train / 8,000 val / 40,000 test
  Treated ratio - Train: 0.385 | Val: 0.391 | Test: 0.384


### 3.4 Model Components


In [12]:
from tensorflow.keras import layers

def torch_style_dense(units, activation=None):
    return layers.Dense(
        units,
        activation=activation,
        kernel_initializer=tf.keras.initializers.VarianceScaling(
            scale=1.0, mode='fan_in', distribution='untruncated_normal'
        ),
        bias_initializer='zeros',
    )

# ---- ShareNetwork: features -> shared representation h (L2-normalized) ----
class ShareNetwork(tf.keras.Model):
    def __init__(self, share_dim=256, base_dim=128, dropout=0.0, normalize='divide', use_batchnorm=False, **kwargs):
        super().__init__(**kwargs)
        self.normalize = normalize
        layers_list = []
        if use_batchnorm:
            layers_list.append(layers.BatchNormalization(momentum=0.9, epsilon=1e-5))
        layers_list.append(torch_style_dense(share_dim, activation='elu'))
        if dropout > 0:
            layers_list.append(layers.Dropout(rate=dropout))
        layers_list.append(torch_style_dense(share_dim, activation='elu'))
        if dropout > 0:
            layers_list.append(layers.Dropout(rate=dropout))
        layers_list.append(torch_style_dense(base_dim, activation='elu'))
        if dropout > 0:
            layers_list.append(layers.Dropout(rate=dropout))
        self.deep_network = tf.keras.Sequential(layers_list)

    def call(self, input_features, training=False):
        shared_representation = self.deep_network(input_features, training=training)
        if self.normalize == 'divide':
            representation_norm = tf.sqrt(
                tf.reduce_sum(tf.square(shared_representation), axis=1, keepdims=True) + 1e-9
            )
            shared_representation = shared_representation / representation_norm
        return shared_representation

# ---- Head: base_dim -> base_dim -> base_dim -> 1 (logit) ----
def make_head(dim=128, dropout=0.0, name='head'):
    layers_list = []
    layers_list.append(torch_style_dense(dim, activation='elu'))
    if dropout > 0:
        layers_list.append(layers.Dropout(rate=dropout))
    layers_list.append(torch_style_dense(dim, activation='elu'))
    if dropout > 0:
        layers_list.append(layers.Dropout(rate=dropout))
    layers_list.append(torch_style_dense(dim, activation='elu'))
    if dropout > 0:
        layers_list.append(layers.Dropout(rate=dropout))
    layers_list.append(torch_style_dense(1))
    return tf.keras.Sequential(layers_list, name=name)

# ---- DESCN / ESX: shared backbone + 4 heads ----
class DESCN(tf.keras.Model):
    """
    Deep Entire Space Cross Networks.
    
    Forward returns 12 tensors:
      propensity_logit, entire_space_treated_response, entire_space_control_response,
      pseudo_effect_logit, treated_response_logit, control_response_logit,
      propensity, treated_response_probability, control_response_probability,
      treated_response_head, control_response_head, shared_representation
    
    The final ITE prediction follows the original code: p_h1 - p_h0 (probability space).
    """
    def __init__(self, input_dim, share_dim=256, base_dim=128,
                 dropout=0.0, normalize='divide', use_batchnorm=False, **kwargs):
        super().__init__(**kwargs)
        self.share_network = ShareNetwork(share_dim, base_dim, dropout, normalize, use_batchnorm)
        self.propensity_head = make_head(base_dim, dropout, 'propensity')
        self.treated_response_head = make_head(base_dim, dropout, 'mu1')
        self.control_response_head = make_head(base_dim, dropout, 'mu0')
        self.pseudo_effect_head = make_head(base_dim, dropout, 'tau')
    
    def call(self, input_features, training=False):
        shared_representation = self.share_network(input_features, training=training)
        
        propensity_logit = self.propensity_head(shared_representation, training=training)
        treated_response_logit = self.treated_response_head(shared_representation, training=training)
        control_response_logit = self.control_response_head(shared_representation, training=training)
        pseudo_effect_logit = self.pseudo_effect_head(shared_representation, training=training)
        
        propensity = tf.clip_by_value(tf.nn.sigmoid(propensity_logit), 1e-3, 1 - 1e-3)
        treated_response_probability = tf.nn.sigmoid(treated_response_logit)
        control_response_probability = tf.nn.sigmoid(control_response_logit)
        
        entire_space_treated_response = propensity * treated_response_probability
        entire_space_control_response = (1.0 - propensity) * control_response_probability
        
        return (
            propensity_logit,
            entire_space_treated_response,
            entire_space_control_response,
            pseudo_effect_logit,
            treated_response_logit,
            control_response_logit,
            propensity,
            treated_response_probability,
            control_response_probability,
            treated_response_probability,
            control_response_probability,
            shared_representation,
        )

print('Model components defined.')


Model components defined.


### 3.5 Loss Functions & IPM Distances


In [13]:
# ---- BCE on probabilities ----
def binary_cross_entropy(labels, predictions):
    predictions = tf.clip_by_value(predictions, 1e-7, 1 - 1e-7)
    return -tf.reduce_mean(labels * tf.math.log(predictions) + (1 - labels) * tf.math.log(1 - predictions))

# ---- Weighted BCE on probabilities ----
def weighted_binary_cross_entropy(labels, predictions, sample_weight):
    predictions = tf.clip_by_value(predictions, 1e-7, 1 - 1e-7)
    per_sample_loss = -(labels * tf.math.log(predictions) + (1 - labels) * tf.math.log(1 - predictions))
    return tf.reduce_mean(sample_weight * per_sample_loss)

# ---- BCE on logits with class weight (for propensity) ----
def weighted_logit_binary_cross_entropy(labels, logits, positive_weight):
    per_sample_loss = tf.nn.sigmoid_cross_entropy_with_logits(labels=labels, logits=logits)
    sample_weight = labels * positive_weight + (1.0 - labels)
    return tf.reduce_mean(sample_weight * per_sample_loss)

def pairwise_squared_distances(left_features, right_features):
    left_squared_norm = tf.reduce_sum(tf.square(left_features), axis=1, keepdims=True)
    right_squared_norm = tf.reduce_sum(tf.square(right_features), axis=1, keepdims=True)
    squared_distances = left_squared_norm - 2.0 * tf.matmul(left_features, right_features, transpose_b=True)
    squared_distances += tf.transpose(right_squared_norm)
    return tf.maximum(squared_distances, 0.0)

def pairwise_euclidean_distances(left_features, right_features):
    return tf.sqrt(tf.maximum(pairwise_squared_distances(left_features, right_features), 1e-9))

def split_treated_control(shared_representations, treatments):
    treatment_vector = tf.reshape(treatments, (-1,))
    treated_indices = tf.where(treatment_vector > 0.5)[:, 0]
    control_indices = tf.where(treatment_vector < 0.5)[:, 0]
    return (
        tf.gather(shared_representations, treated_indices),
        tf.gather(shared_representations, control_indices),
    )

def energy_distance_between_groups(shared_representations, treatments):
    treated_representations, control_representations = split_treated_control(shared_representations, treatments)

    def compute_distance():
        cross_distance = tf.reduce_mean(pairwise_euclidean_distances(treated_representations, control_representations))
        treated_distance = tf.reduce_mean(pairwise_euclidean_distances(treated_representations, treated_representations))
        control_distance = tf.reduce_mean(pairwise_euclidean_distances(control_representations, control_representations))
        return tf.maximum(2.0 * cross_distance - treated_distance - control_distance, 0.0)

    has_both_groups = tf.logical_and(tf.shape(treated_representations)[0] > 0, tf.shape(control_representations)[0] > 0)
    return tf.cond(has_both_groups, compute_distance, lambda: tf.constant(0.0, dtype=shared_representations.dtype))

def sinkhorn_transport_cost(left_features, right_features, epsilon=0.05, iterations=30):
    cost_matrix = pairwise_squared_distances(left_features, right_features)
    left_count = tf.shape(left_features)[0]
    right_count = tf.shape(right_features)[0]
    log_left_weights = -tf.math.log(tf.cast(left_count, cost_matrix.dtype)) * tf.ones((left_count,), dtype=cost_matrix.dtype)
    log_right_weights = -tf.math.log(tf.cast(right_count, cost_matrix.dtype)) * tf.ones((right_count,), dtype=cost_matrix.dtype)
    log_kernel = -cost_matrix / epsilon
    log_u = tf.zeros_like(log_left_weights)
    log_v = tf.zeros_like(log_right_weights)

    for _ in range(iterations):
        log_u = log_left_weights - tf.reduce_logsumexp(log_kernel + log_v[tf.newaxis, :], axis=1)
        log_v = log_right_weights - tf.reduce_logsumexp(log_kernel + log_u[:, tf.newaxis], axis=0)

    transport_plan = tf.exp(log_u[:, tf.newaxis] + log_kernel + log_v[tf.newaxis, :])
    return tf.reduce_sum(transport_plan * cost_matrix)

def sinkhorn_distance_between_groups(shared_representations, treatments):
    treated_representations, control_representations = split_treated_control(shared_representations, treatments)

    def compute_distance():
        # Approximation of GeomLoss SamplesLoss(loss="sinkhorn", p=2, blur=0.05).
        cross_cost = sinkhorn_transport_cost(treated_representations, control_representations)
        treated_self_cost = sinkhorn_transport_cost(treated_representations, treated_representations)
        control_self_cost = sinkhorn_transport_cost(control_representations, control_representations)
        return tf.maximum(cross_cost - 0.5 * treated_self_cost - 0.5 * control_self_cost, 0.0)

    has_both_groups = tf.logical_and(tf.shape(treated_representations)[0] > 0, tf.shape(control_representations)[0] > 0)
    return tf.cond(has_both_groups, compute_distance, lambda: tf.constant(0.0, dtype=shared_representations.dtype))

# ---- IPM distances ----
def wasserstein_distance(shared_representations, treatments):
    return sinkhorn_distance_between_groups(shared_representations, treatments)

def mmd_distance(shared_representations, treatments):
    # Original code names this CFRmmd, but GeomLoss uses SamplesLoss(loss="energy").
    return energy_distance_between_groups(shared_representations, treatments)

def masked_binary_cross_entropy(labels, predictions, mask):
    return tf.cond(
        tf.reduce_any(mask),
        lambda: binary_cross_entropy(tf.boolean_mask(labels, mask), tf.boolean_mask(predictions, mask)),
        lambda: tf.constant(0.0, dtype=predictions.dtype),
    )

print('Loss functions defined.')


Loss functions defined.


### 3.6 Evaluation


In [14]:
from sklift.metrics import qini_auc_score as sklift_qini_auc_score


def qini_auc_score(y, p_tau, t):
    """Match the public DESCN evaluator by delegating to scikit-uplift."""
    return float(sklift_qini_auc_score(
        np.asarray(y).reshape(-1),
        np.asarray(p_tau).reshape(-1),
        np.asarray(t).reshape(-1),
    ))

def evaluate(
    model,
    features,
    outcomes,
    treatments,
    randomized_flags,
    treatment_effect_true=None,
    weights=None,
    compute_loss=True,
    compute_imbalance_loss=False,
):
    """Compute paper metrics and optional training-style losses.

    IPM imbalance loss is a training regularizer. Computing it on the full
    validation/test set creates huge pairwise matrices, so evaluation skips it
    unless explicitly requested.
    """
    loss_weights = weights or {}
    propensity_weight = loss_weights.get('prpsy_w', 0)
    entire_treated_weight = loss_weights.get('escvr1_w', 0)
    entire_control_weight = loss_weights.get('escvr0_w', 0)
    treated_response_weight = loss_weights.get('h1_w', 0)
    control_response_weight = loss_weights.get('h0_w', 0)
    cross_treated_weight = loss_weights.get('mu1hat_w', 0)
    cross_control_weight = loss_weights.get('mu0hat_w', 0)
    imbalance_weight = loss_weights.get('imb_dist_w', 0)
    imbalance_type = loss_weights.get('imb_dist', 'wass')
    reweight_sample = loss_weights.get('reweight_sample', True)
    
    (
        propensity_logit,
        entire_space_treated_response,
        entire_space_control_response,
        pseudo_effect_logit,
        treated_response_logit,
        control_response_logit,
        _,
        _,
        _,
        treated_response_head,
        control_response_head,
        shared_representation,
    ) = model(features, training=False)
    
    treatment_rate = tf.reduce_mean(treatments)
    if reweight_sample:
        sample_weight = treatments / (2 * treatment_rate) + (1 - treatments) / (2 * (1 - treatment_rate))
    else:
        sample_weight = tf.ones_like(treatments)
    
    non_randomized_mask = tf.reshape(tf.cast(~tf.cast(randomized_flags, tf.bool), tf.bool), (-1,))
    
    def mask_non_randomized(values):
        return tf.boolean_mask(values, non_randomized_mask)

    losses = {}
    if compute_loss:
        masked_sample_weight = tf.reshape(tf.boolean_mask(tf.reshape(sample_weight, (-1,)), non_randomized_mask), (-1, 1))
        
        if propensity_weight > 0:
            positive_weight = 1.0 / (2.0 * tf.maximum(treatment_rate, 1e-5))
            losses['propensity'] = propensity_weight * weighted_logit_binary_cross_entropy(
                mask_non_randomized(treatments),
                mask_non_randomized(propensity_logit),
                positive_weight,
            )
        if entire_treated_weight > 0:
            losses['estr'] = entire_treated_weight * weighted_binary_cross_entropy(
                mask_non_randomized(outcomes * treatments),
                mask_non_randomized(entire_space_treated_response),
                masked_sample_weight,
            )
        if entire_control_weight > 0:
            losses['escr'] = entire_control_weight * weighted_binary_cross_entropy(
                mask_non_randomized(outcomes * (1 - treatments)),
                mask_non_randomized(entire_space_control_response),
                masked_sample_weight,
            )
        
        treated_mask = treatments[:, 0] > 0.5
        control_mask = ~treated_mask
        if treated_response_weight > 0:
            losses['tr'] = treated_response_weight * masked_binary_cross_entropy(
                outcomes, treated_response_head, treated_mask
            )
        if control_response_weight > 0:
            losses['cr'] = control_response_weight * masked_binary_cross_entropy(
                outcomes, control_response_head, control_mask
            )
        
        if cross_treated_weight > 0:
            cross_treated_response = tf.nn.sigmoid(control_response_logit + pseudo_effect_logit)
            losses['cross_tr'] = cross_treated_weight * masked_binary_cross_entropy(
                outcomes, cross_treated_response, treated_mask
            )
        if cross_control_weight > 0:
            cross_control_response = tf.nn.sigmoid(treated_response_logit - pseudo_effect_logit)
            losses['cross_cr'] = cross_control_weight * masked_binary_cross_entropy(
                outcomes, cross_control_response, control_mask
            )
        
        if compute_imbalance_loss and imbalance_weight > 0:
            imbalance_loss = wasserstein_distance(shared_representation, treatments)
            if imbalance_type == 'mmd':
                imbalance_loss = mmd_distance(shared_representation, treatments)
            losses['imb'] = imbalance_weight * imbalance_loss
    
    total_loss = tf.add_n(list(losses.values())) if losses else tf.constant(0.0)
    
    propensity_predicted = tf.nn.sigmoid(propensity_logit).numpy()
    treatment_effect_predicted = (treated_response_head - control_response_head).numpy()
    outcomes_numpy = outcomes.numpy()
    treatments_numpy = treatments.numpy()
    factual_outcome_predicted = (treated_response_head * treatments + control_response_head * (1 - treatments)).numpy()
    counterfactual_outcome_predicted = (control_response_head * treatments + treated_response_head * (1 - treatments)).numpy()
    
    results = {
        'total_loss': float(total_loss.numpy()),
        'p_prpsy': propensity_predicted,
        'p_yf': factual_outcome_predicted,
        'p_ycf': counterfactual_outcome_predicted,
        'p_tau': treatment_effect_predicted,
    }
    try:
        results['auuc'] = qini_auc_score(
            outcomes_numpy,
            treatment_effect_predicted,
            treatments_numpy,
        )
    except Exception:
        results['auuc'] = 0.0
    
    flattened_treatments = treatments_numpy.flatten()
    if treatment_effect_true is not None:
        treatment_effect_true_numpy = treatment_effect_true.numpy()
        results['sqrt_pehe'] = np.sqrt(np.mean(np.square(treatment_effect_predicted - treatment_effect_true_numpy)))
        results['e_ate'] = np.abs(treatment_effect_predicted.mean() - treatment_effect_true_numpy.mean())
        if np.any(flattened_treatments == 1):
            results['e_att'] = np.abs(
                treatment_effect_predicted[flattened_treatments == 1].mean()
                - treatment_effect_true_numpy[flattened_treatments == 1].mean()
            )
    elif np.any(flattened_treatments == 1) and np.any(flattened_treatments == 0):
        flattened_outcomes = outcomes_numpy.flatten()
        observed_att = flattened_outcomes[flattened_treatments == 1].mean() - flattened_outcomes[flattened_treatments == 0].mean()
        results['e_att'] = np.abs(treatment_effect_predicted[flattened_treatments == 1].mean() - observed_att)
    
    for loss_name, loss_value in losses.items():
        results[f'loss_{loss_name}'] = float(loss_value.numpy())
    return results

print('evaluate() defined.')


evaluate() defined.


### 3.7 Training


In [15]:
from pathlib import Path

def save_test_prediction_artifacts(model_name, experiment_predictions):
    """Save public-code-style test prediction artifacts: units x experiment x output."""
    output_dir = Path(TRAIN_CONFIG.get('prediction_output_dir', 'results/synthetic'))
    output_dir.mkdir(parents=True, exist_ok=True)

    prediction_arrays = {}
    for prediction_name in ['p_prpsy', 'p_yf', 'p_ycf', 'p_tau']:
        prediction_arrays[prediction_name] = np.stack(
            [experiment_result[prediction_name] for experiment_result in experiment_predictions],
            axis=1,
        )

    loss_by_experiment = np.array([experiment_result['loss'] for experiment_result in experiment_predictions])
    prediction_arrays['loss'] = np.transpose(loss_by_experiment, (1, 2, 0))
    prediction_arrays['val'] = np.array([])

    output_path = output_dir / f'{model_name}_test_result.test'
    np.savez(output_path, **prediction_arrays)
    print(f'Saved test predictions: {output_path}.npz')


def train_step(model, features_batch, treatments_batch, outcomes_batch, randomized_flags_batch, loss_weights, optimizer):
    with tf.GradientTape() as tape:
        (
            propensity_logit,
            entire_space_treated_response,
            entire_space_control_response,
            pseudo_effect_logit,
            treated_response_logit,
            control_response_logit,
            _,
            _,
            _,
            treated_response_head,
            control_response_head,
            shared_representation,
        ) = model(features_batch, training=True)
        
        treatment_rate = tf.reduce_mean(treatments_batch)
        if loss_weights.get('reweight_sample', True):
            sample_weight = treatments_batch / (2 * treatment_rate) + (1 - treatments_batch) / (2 * (1 - treatment_rate))
        else:
            sample_weight = tf.ones_like(treatments_batch)
        
        non_randomized_mask = tf.reshape(tf.cast(~tf.cast(randomized_flags_batch, tf.bool), tf.bool), (-1,))

        def mask_non_randomized(values):
            return tf.boolean_mask(values, non_randomized_mask)

        masked_sample_weight = tf.reshape(tf.boolean_mask(tf.reshape(sample_weight, (-1,)), non_randomized_mask), (-1, 1))
        
        total_loss = tf.constant(0.0)
        if loss_weights['prpsy_w'] > 0:
            positive_weight = 1.0 / (2.0 * tf.maximum(treatment_rate, 1e-5))
            total_loss += loss_weights['prpsy_w'] * weighted_logit_binary_cross_entropy(
                mask_non_randomized(treatments_batch),
                mask_non_randomized(propensity_logit),
                positive_weight,
            )
        if loss_weights['escvr1_w'] > 0:
            total_loss += loss_weights['escvr1_w'] * weighted_binary_cross_entropy(
                mask_non_randomized(outcomes_batch * treatments_batch),
                mask_non_randomized(entire_space_treated_response),
                masked_sample_weight,
            )
        if loss_weights['escvr0_w'] > 0:
            total_loss += loss_weights['escvr0_w'] * weighted_binary_cross_entropy(
                mask_non_randomized(outcomes_batch * (1 - treatments_batch)),
                mask_non_randomized(entire_space_control_response),
                masked_sample_weight,
            )
        
        treated_mask = treatments_batch[:, 0] > 0.5
        control_mask = ~treated_mask
        if loss_weights['h1_w'] > 0:
            total_loss += loss_weights['h1_w'] * masked_binary_cross_entropy(
                outcomes_batch, treated_response_head, treated_mask
            )
        if loss_weights['h0_w'] > 0:
            total_loss += loss_weights['h0_w'] * masked_binary_cross_entropy(
                outcomes_batch, control_response_head, control_mask
            )
        
        if loss_weights['mu1hat_w'] > 0:
            cross_treated_response = tf.nn.sigmoid(control_response_logit + pseudo_effect_logit)
            total_loss += loss_weights['mu1hat_w'] * masked_binary_cross_entropy(
                outcomes_batch, cross_treated_response, treated_mask
            )
        if loss_weights['mu0hat_w'] > 0:
            cross_control_response = tf.nn.sigmoid(treated_response_logit - pseudo_effect_logit)
            total_loss += loss_weights['mu0hat_w'] * masked_binary_cross_entropy(
                outcomes_batch, cross_control_response, control_mask
            )
        
        if loss_weights['imb_dist_w'] > 0:
            imbalance_loss = wasserstein_distance(shared_representation, treatments_batch)
            if loss_weights['imb_dist'] == 'mmd':
                imbalance_loss = mmd_distance(shared_representation, treatments_batch)
            total_loss += loss_weights['imb_dist_w'] * imbalance_loss
    
    gradients = tape.gradient(total_loss, model.trainable_variables)
    grads_and_vars = [(g, v) for g, v in zip(gradients, model.trainable_variables) if g is not None]
    optimizer.apply_gradients(grads_and_vars)
    return total_loss


def train(model, features_train, outcomes_train, treatments_train, randomized_flags_train,
          features_validation, outcomes_validation, treatments_validation, randomized_flags_validation,
          features_test, outcomes_test, treatments_test, randomized_flags_test, treatment_effect_test=None,
          weights=None, epochs=15, batch_size=500, lr=1e-3, l2=1e-2, name='Model'):
    """Train one model. Returns history, selected metrics, and test prediction artifact arrays."""
    loss_weights = dict(weights or {})
    loss_weights.setdefault('reweight_sample', True)
    loss_weights.setdefault('imb_dist', 'wass')
    for weight_name in ['prpsy_w','escvr1_w','escvr0_w','h1_w','h0_w','mu1hat_w','mu0hat_w','imb_dist_w']:
        loss_weights.setdefault(weight_name, 0.0)
    
    train_dataset = tf.data.Dataset.from_tensor_slices(
        (features_train, treatments_train, outcomes_train, randomized_flags_train)
    )
    train_dataset = train_dataset.shuffle(len(features_train)).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr, weight_decay=l2)
    
    history = {'epoch': [], 'lr': [], 'train_loss': [], 'val_loss': [],
               'auuc': [], 'e_att': [], 'sqrt_pehe': [], 'e_ate': []}
    selection_metric = TRAIN_CONFIG.get('selection_metric', 'auuc')
    test_prediction_history = {'p_prpsy': [], 'p_yf': [], 'p_ycf': [], 'p_tau': [], 'loss': []}
    
    print(f'[{name}] epochs={epochs}  batch={batch_size}  lr={lr}  l2={l2}  select={selection_metric}')
    print(f'  weights: prpsy={loss_weights["prpsy_w"]} estr={loss_weights["escvr1_w"]} escr={loss_weights["escvr0_w"]} '
          f'h1={loss_weights["h1_w"]} h0={loss_weights["h0_w"]} '
          f'xTR={loss_weights["mu1hat_w"]} xCR={loss_weights["mu0hat_w"]} imb={loss_weights["imb_dist_w"]}')
    print(f'  train: {len(features_train):,}  val: {len(features_validation):,}  test: {len(features_test):,}  '
          f'treated(train): {tf.reduce_mean(treatments_train).numpy():.3f}')
    print('-' * 60)
    
    for epoch_index in tqdm(range(epochs), desc=name, unit="ep"):
        decay_exponent = epoch_index // TRAIN_CONFIG['decay_step_size']
        current_lr = lr * (TRAIN_CONFIG['decay_rate'] ** decay_exponent)
        optimizer.learning_rate.assign(current_lr)
        epoch_losses = []
        for features_batch, treatments_batch, outcomes_batch, randomized_flags_batch in train_dataset:
            if tf.shape(features_batch)[0] < batch_size:
                continue
            batch_loss = train_step(
                model,
                features_batch,
                treatments_batch,
                outcomes_batch,
                randomized_flags_batch,
                loss_weights,
                optimizer,
            )
            epoch_losses.append(float(batch_loss.numpy()))
        
        average_loss = np.mean(epoch_losses) if epoch_losses else 0.0
        validation_results = evaluate(
            model,
            features_validation,
            outcomes_validation,
            treatments_validation,
            randomized_flags_validation,
            weights=loss_weights,
            compute_loss=True,
            compute_imbalance_loss=False,
        )
        test_results = evaluate(
            model,
            features_test,
            outcomes_test,
            treatments_test,
            randomized_flags_test,
            treatment_effect_true=treatment_effect_test,
            weights=loss_weights,
            compute_loss=False,
        )
        
        history['epoch'].append(epoch_index)
        history['lr'].append(float(current_lr))
        history['train_loss'].append(average_loss)
        history['val_loss'].append(validation_results['total_loss'])
        history['auuc'].append(test_results.get('auuc', 0))
        history['e_att'].append(test_results.get('e_att', 0))
        history['sqrt_pehe'].append(test_results.get('sqrt_pehe', 0))
        history['e_ate'].append(test_results.get('e_ate', 0))
        for prediction_name in ['p_prpsy', 'p_yf', 'p_ycf', 'p_tau']:
            test_prediction_history[prediction_name].append(test_results[prediction_name].reshape(-1))
        test_prediction_history['loss'].append([test_results['total_loss']])
        
    
    print('-' * 60)
    if history[selection_metric]:
        metric_values = np.asarray(history[selection_metric], dtype=float)
        if selection_metric in ('sqrt_pehe', 'e_ate', 'e_att', 'val_loss'):
            selected_epoch_index = int(np.argmin(metric_values))
        else:
            selected_epoch_index = int(np.argmax(metric_values))
        selected_test_results = {
            'selection': f'best_{selection_metric}',
            'selected_epoch': int(history['epoch'][selected_epoch_index]),
            'auuc': float(history['auuc'][selected_epoch_index]),
            'e_att': float(history['e_att'][selected_epoch_index]),
            'sqrt_pehe': float(history['sqrt_pehe'][selected_epoch_index]),
            'e_ate': float(history['e_ate'][selected_epoch_index]),
        }
    else:
        selected_test_results = {
            'selection': f'best_{selection_metric}',
            'selected_epoch': -1,
            'auuc': 0.0,
            'e_att': 0.0,
            'sqrt_pehe': 0.0,
            'e_ate': 0.0,
        }

    print(f'[{name}] Best {selection_metric} epoch={selected_test_results["selected_epoch"] + 1}/{epochs}: '
          f'AUUC={selected_test_results["auuc"]:.4f}  '
          f'e_ATT={selected_test_results["e_att"]:.4f}  '
          f'sqrtPEHE={selected_test_results["sqrt_pehe"]:.4f}  '
          f'e_ATE={selected_test_results["e_ate"]:.4f}')
    
    test_prediction_artifact = {
        prediction_name: np.stack(values, axis=1)
        for prediction_name, values in test_prediction_history.items()
        if prediction_name != 'loss'
    }
    test_prediction_artifact['loss'] = np.array(test_prediction_history['loss'])

    return history, selected_test_results, test_prediction_artifact

print('train() defined.')


train() defined.


---


## 4. Model Training

Each model uses the same TensorFlow implementation of the DESCN/ESX backbone. The model name is determined by the loss weights in `MODEL_CONFIGS`.

Selection rule:

- ACIC: choose the epoch with lowest `sqrt_pehe`, matching `DESCN/eval.py`.
- Lazada: choose the epoch with highest AUUC, matching `DESCN/eval4real_data.py`.

### 4.1 TARNet

Shared representation with treated and control response heads. Active losses: direct TR/CR losses only.

In [16]:
def train_tarnet():
    """Train TARNet model. Returns (results, histories, sqrt_pehe, e_ate, auuc)."""
    seed_everything(2)
    loss_weights = MODEL_CONFIGS['TARNet']

    tarnet_results = []
    histories = []
    prediction_artifacts = []

    for experiment_index in range(EXPERIMENT_COUNT):
        experiment_data = make_experiment_tensors(experiment_index)
        tarnet_model = DESCN(
            input_dim=training_data['x'].shape[1],
            share_dim=HYPERPARAMS['share_dim'],
            base_dim=HYPERPARAMS['base_dim'],
            dropout=HYPERPARAMS['dropout'],
            use_batchnorm=HYPERPARAMS['use_batchnorm'],
        )
        _ = tarnet_model(tf.zeros((2, training_data['x'].shape[1])), training=False)
        
        history, selected_metrics, test_prediction_artifact = train(
            tarnet_model,
            experiment_data['features_train'],
            experiment_data['outcomes_train'],
            experiment_data['treatments_train'],
            experiment_data['randomized_flags_train'],
            experiment_data['features_validation'],
            experiment_data['outcomes_validation'],
            experiment_data['treatments_validation'],
            experiment_data['randomized_flags_validation'],
            experiment_data['features_test'],
            experiment_data['outcomes_test'],
            experiment_data['treatments_test'],
            experiment_data['randomized_flags_test'],
            experiment_data['treatment_effect_test'],
            weights=loss_weights,
            epochs=TRAIN_CONFIG['epochs'],
            batch_size=HYPERPARAMS['batch_size'],
            lr=HYPERPARAMS['lr'],
            l2=HYPERPARAMS['l2'],
            name=f'TARNet exp {experiment_index + 1}/{EXPERIMENT_COUNT}',
        )
        tarnet_results.append(selected_metrics)
        histories.append(history)
        prediction_artifacts.append(test_prediction_artifact)
        del tarnet_model, experiment_data
        tf.keras.backend.clear_session()
        gc.collect()

    save_test_prediction_artifacts('TARNet_tf', prediction_artifacts)

    sqrt_pehe_values = np.array([result.get('sqrt_pehe', 0.0) for result in tarnet_results])
    e_ate_values = np.array([result.get('e_ate', 0.0) for result in tarnet_results])
    auuc_values = np.array([result['auuc'] for result in tarnet_results])
    e_att_values = np.array([result.get('e_att', 0.0) for result in tarnet_results])
    print(f'TARNet ({EXPERIMENT_COUNT} experiments): sqrtPEHE={sqrt_pehe_values.mean():.4f}+/-{sqrt_pehe_values.std()/EXPERIMENT_COUNT**0.5:.4f}  '
          f'e_ATE={e_ate_values.mean():.4f}+/-{e_ate_values.std()/EXPERIMENT_COUNT**0.5:.4f}  '
          f'AUUC={auuc_values.mean():.4f}+/-{auuc_values.std()/EXPERIMENT_COUNT**0.5:.4f}  '
          f'e_ATT={e_att_values.mean():.4f}+/-{e_att_values.std()/EXPERIMENT_COUNT**0.5:.4f}')
    return tarnet_results, histories, sqrt_pehe_values, e_ate_values, auuc_values, e_att_values

### 4.2 CFR(MMD)

TARNet plus an IPM penalty on the shared representation. This notebook uses the MMD configuration from the DESCN source.

In [17]:
def train_cfr_mmd():
    """Train CFR(MMD) model. Returns (results, histories, sqrt_pehe, e_ate, auuc)."""
    seed_everything(2)
    loss_weights = MODEL_CONFIGS['CFR_MMD']

    cfr_mmd_results = []
    histories = []
    prediction_artifacts = []

    for experiment_index in range(EXPERIMENT_COUNT):
        experiment_data = make_experiment_tensors(experiment_index)
        cfr_mmd_model = DESCN(
            input_dim=training_data['x'].shape[1],
            share_dim=HYPERPARAMS['share_dim'],
            base_dim=HYPERPARAMS['base_dim'],
            dropout=HYPERPARAMS['dropout'],
            use_batchnorm=HYPERPARAMS['use_batchnorm'],
        )
        _ = cfr_mmd_model(tf.zeros((2, training_data['x'].shape[1])), training=False)
        
        history, selected_metrics, test_prediction_artifact = train(
            cfr_mmd_model,
            experiment_data['features_train'],
            experiment_data['outcomes_train'],
            experiment_data['treatments_train'],
            experiment_data['randomized_flags_train'],
            experiment_data['features_validation'],
            experiment_data['outcomes_validation'],
            experiment_data['treatments_validation'],
            experiment_data['randomized_flags_validation'],
            experiment_data['features_test'],
            experiment_data['outcomes_test'],
            experiment_data['treatments_test'],
            experiment_data['randomized_flags_test'],
            experiment_data['treatment_effect_test'],
            weights=loss_weights,
            epochs=TRAIN_CONFIG['epochs'],
            batch_size=HYPERPARAMS['batch_size'],
            lr=HYPERPARAMS['lr'],
            l2=HYPERPARAMS['l2'],
            name=f'CFR(MMD) exp {experiment_index + 1}/{EXPERIMENT_COUNT}',
        )
        cfr_mmd_results.append(selected_metrics)
        histories.append(history)
        prediction_artifacts.append(test_prediction_artifact)
        del cfr_mmd_model, experiment_data
        tf.keras.backend.clear_session()
        gc.collect()

    save_test_prediction_artifacts('CFR_mmd_tf', prediction_artifacts)

    sqrt_pehe_values = np.array([result.get('sqrt_pehe', 0.0) for result in cfr_mmd_results])
    e_ate_values = np.array([result.get('e_ate', 0.0) for result in cfr_mmd_results])
    auuc_values = np.array([result['auuc'] for result in cfr_mmd_results])
    e_att_values = np.array([result.get('e_att', 0.0) for result in cfr_mmd_results])
    print(f'CFR(MMD) ({EXPERIMENT_COUNT} experiments): sqrtPEHE={sqrt_pehe_values.mean():.4f}+/-{sqrt_pehe_values.std()/EXPERIMENT_COUNT**0.5:.4f}  '
          f'e_ATE={e_ate_values.mean():.4f}+/-{e_ate_values.std()/EXPERIMENT_COUNT**0.5:.4f}  '
          f'AUUC={auuc_values.mean():.4f}+/-{auuc_values.std()/EXPERIMENT_COUNT**0.5:.4f}  '
          f'e_ATT={e_att_values.mean():.4f}+/-{e_att_values.std()/EXPERIMENT_COUNT**0.5:.4f}')
    return cfr_mmd_results, histories, sqrt_pehe_values, e_ate_values, auuc_values, e_att_values

### 4.3 X-network

Adds the pseudo treatment-effect head and cross-response losses. ESN losses are inactive.

In [18]:
def train_xnetwork():
    """Train X-network model. Returns (results, histories, sqrt_pehe, e_ate, auuc)."""
    seed_everything(2)
    loss_weights = MODEL_CONFIGS['X_network']

    xnetwork_results = []
    histories = []
    prediction_artifacts = []

    for experiment_index in range(EXPERIMENT_COUNT):
        experiment_data = make_experiment_tensors(experiment_index)
        xnetwork_model = DESCN(
            input_dim=training_data['x'].shape[1],
            share_dim=HYPERPARAMS['share_dim'],
            base_dim=HYPERPARAMS['base_dim'],
            dropout=HYPERPARAMS['dropout'],
            use_batchnorm=HYPERPARAMS['use_batchnorm'],
        )
        _ = xnetwork_model(tf.zeros((2, training_data['x'].shape[1])), training=False)
        
        history, selected_metrics, test_prediction_artifact = train(
            xnetwork_model,
            experiment_data['features_train'],
            experiment_data['outcomes_train'],
            experiment_data['treatments_train'],
            experiment_data['randomized_flags_train'],
            experiment_data['features_validation'],
            experiment_data['outcomes_validation'],
            experiment_data['treatments_validation'],
            experiment_data['randomized_flags_validation'],
            experiment_data['features_test'],
            experiment_data['outcomes_test'],
            experiment_data['treatments_test'],
            experiment_data['randomized_flags_test'],
            experiment_data['treatment_effect_test'],
            weights=loss_weights,
            epochs=TRAIN_CONFIG['epochs'],
            batch_size=HYPERPARAMS['batch_size'],
            lr=HYPERPARAMS['lr'],
            l2=HYPERPARAMS['l2'],
            name=f'X-network exp {experiment_index + 1}/{EXPERIMENT_COUNT}',
        )
        xnetwork_results.append(selected_metrics)
        histories.append(history)
        prediction_artifacts.append(test_prediction_artifact)
        del xnetwork_model, experiment_data
        tf.keras.backend.clear_session()
        gc.collect()

    save_test_prediction_artifacts('Xnetwork_tf', prediction_artifacts)

    sqrt_pehe_values = np.array([result.get('sqrt_pehe', 0.0) for result in xnetwork_results])
    e_ate_values = np.array([result.get('e_ate', 0.0) for result in xnetwork_results])
    auuc_values = np.array([result['auuc'] for result in xnetwork_results])
    e_att_values = np.array([result.get('e_att', 0.0) for result in xnetwork_results])
    print(f'X-network ({EXPERIMENT_COUNT} experiments): sqrtPEHE={sqrt_pehe_values.mean():.4f}+/-{sqrt_pehe_values.std()/EXPERIMENT_COUNT**0.5:.4f}  '
          f'e_ATE={e_ate_values.mean():.4f}+/-{e_ate_values.std()/EXPERIMENT_COUNT**0.5:.4f}  '
          f'AUUC={auuc_values.mean():.4f}+/-{auuc_values.std()/EXPERIMENT_COUNT**0.5:.4f}  '
          f'e_ATT={e_att_values.mean():.4f}+/-{e_att_values.std()/EXPERIMENT_COUNT**0.5:.4f}')
    return xnetwork_results, histories, sqrt_pehe_values, e_ate_values, auuc_values, e_att_values

### 4.4 DESCN

Combines ESN losses and X-network cross losses. Direct TR/CR losses are inactive in the DESCN configuration.

In [19]:
def train_descn():
    """Train DESCN model. Returns (results, histories, sqrt_pehe, e_ate, auuc)."""
    seed_everything(2)
    loss_weights = MODEL_CONFIGS['DESCN']

    descn_results = []
    histories = []
    prediction_artifacts = []

    for experiment_index in range(EXPERIMENT_COUNT):
        experiment_data = make_experiment_tensors(experiment_index)
        descn_model = DESCN(
            input_dim=training_data['x'].shape[1],
            share_dim=HYPERPARAMS['share_dim'],
            base_dim=HYPERPARAMS['base_dim'],
            dropout=HYPERPARAMS['dropout'],
            use_batchnorm=HYPERPARAMS['use_batchnorm'],
        )
        _ = descn_model(tf.zeros((2, training_data['x'].shape[1])), training=False)
        
        history, selected_metrics, test_prediction_artifact = train(
            descn_model,
            experiment_data['features_train'],
            experiment_data['outcomes_train'],
            experiment_data['treatments_train'],
            experiment_data['randomized_flags_train'],
            experiment_data['features_validation'],
            experiment_data['outcomes_validation'],
            experiment_data['treatments_validation'],
            experiment_data['randomized_flags_validation'],
            experiment_data['features_test'],
            experiment_data['outcomes_test'],
            experiment_data['treatments_test'],
            experiment_data['randomized_flags_test'],
            experiment_data['treatment_effect_test'],
            weights=loss_weights,
            epochs=TRAIN_CONFIG['epochs'],
            batch_size=HYPERPARAMS['batch_size'],
            lr=HYPERPARAMS['lr'],
            l2=HYPERPARAMS['l2'],
            name=f'DESCN exp {experiment_index + 1}/{EXPERIMENT_COUNT}',
        )
        descn_results.append(selected_metrics)
        histories.append(history)
        prediction_artifacts.append(test_prediction_artifact)
        del descn_model, experiment_data
        tf.keras.backend.clear_session()
        gc.collect()

    save_test_prediction_artifacts('DESCN_tf', prediction_artifacts)

    sqrt_pehe_values = np.array([result.get('sqrt_pehe', 0.0) for result in descn_results])
    e_ate_values = np.array([result.get('e_ate', 0.0) for result in descn_results])
    auuc_values = np.array([result['auuc'] for result in descn_results])
    e_att_values = np.array([result.get('e_att', 0.0) for result in descn_results])
    print(f'DESCN ({EXPERIMENT_COUNT} experiments): sqrtPEHE={sqrt_pehe_values.mean():.4f}+/-{sqrt_pehe_values.std()/EXPERIMENT_COUNT**0.5:.4f}  '
          f'e_ATE={e_ate_values.mean():.4f}+/-{e_ate_values.std()/EXPERIMENT_COUNT**0.5:.4f}  '
          f'AUUC={auuc_values.mean():.4f}+/-{auuc_values.std()/EXPERIMENT_COUNT**0.5:.4f}  '
          f'e_ATT={e_att_values.mean():.4f}+/-{e_att_values.std()/EXPERIMENT_COUNT**0.5:.4f}')
    return descn_results, histories, sqrt_pehe_values, e_ate_values, auuc_values, e_att_values

---


## 5. Results Comparison

ACIC uses ground-truth treatment effects, so the paper reports PEHE and ATE error:

$$\epsilon_{PEHE}=\sqrt{\frac{1}{n}\sum_i(\hat{\mu}_1(x_i)-\hat{\mu}_0(x_i)-\tau(x_i))^2}$$
$$\epsilon_{ATE}=\left|\frac{1}{n}\sum_i(\hat{\mu}_1(x_i)-\hat{\mu}_0(x_i))-\frac{1}{n}\sum_i\tau(x_i)\right|$$

Lazada has no individual ground-truth treatment effect, so the paper reports AUUC and ATT error on the randomized test set:

$$ATT=\frac{1}{|T|}\sum_{i\in T}y_i-\frac{1}{|C|}\sum_{i\in C}y_i$$
$$\epsilon_{ATT}=\left|\frac{1}{|T|}\sum_{i\in T}(\hat{\mu}_1(x_i)-\hat{\mu}_0(x_i))-ATT\right|$$

Relative improvement is computed against CFR(MMD):

$$\text{Improvement}=\frac{E_{CFR(MMD)}-E_{model}}{E_{CFR(MMD)}}\times100\%$$

for error metrics, and

$$\text{Improvement}=\frac{AUUC_{model}-AUUC_{CFR(MMD)}}{AUUC_{CFR(MMD)}}\times100\%$$

for AUUC.

In [ ]:
import matplotlib.pyplot as plt

# Train all models
tarnet_results, tarnet_histories, tarnet_sqrt_pehe, tarnet_e_ate, tarnet_auuc, tarnet_e_att = train_tarnet()
cfr_mmd_results, cfr_mmd_histories, cfr_mmd_sqrt_pehe, cfr_mmd_e_ate, cfr_mmd_auuc, cfr_mmd_e_att = train_cfr_mmd()
xnetwork_results, xnetwork_histories, xnetwork_sqrt_pehe, xnetwork_e_ate, xnetwork_auuc, xnetwork_e_att = train_xnetwork()
descn_results, descn_histories, descn_sqrt_pehe, descn_e_ate, descn_auuc, descn_e_att = train_descn()

baseline_model_name = 'CFR(MMD)'
model_names = ['TARNet', baseline_model_name, 'X-network', 'DESCN']
sqrt_pehe_values = [tarnet_sqrt_pehe, cfr_mmd_sqrt_pehe, xnetwork_sqrt_pehe, descn_sqrt_pehe]
e_ate_values = [tarnet_e_ate, cfr_mmd_e_ate, xnetwork_e_ate, descn_e_ate]
auuc_values = [tarnet_auuc, cfr_mmd_auuc, xnetwork_auuc, descn_auuc]
e_att_values = [tarnet_e_att, cfr_mmd_e_att, xnetwork_e_att, descn_e_att]

has_ground_truth = training_data['tau'] is not None
baseline_sqrt_pehe = cfr_mmd_sqrt_pehe.mean() if has_ground_truth else 0
baseline_auuc = cfr_mmd_auuc.mean()
baseline_e_att = cfr_mmd_e_att.mean()

print('=' * 110)
print(f'MODEL COMPARISON ({EXPERIMENT_COUNT} experiments, best {TRAIN_CONFIG["selection_metric"]} epoch, mean +/- std error)')
print('=' * 110)
if has_ground_truth:
    header = f'{"Model":<16s} {"sqrtPEHE":<22s} {"Impr%":>8s}  {"e_ATE":<22s}'
else:
    header = f'{"Model":<16s} {"AUUC":<22s} {"AUUC Impr%":>11s} {"e_ATT":<22s} {"e_ATT Impr%":>12s}'
print(header)
print('-' * 110)

for model_name, sq, ea, au, ev in zip(model_names, sqrt_pehe_values, e_ate_values, auuc_values, e_att_values):
    if has_ground_truth:
        sq_mean, sq_se = sq.mean(), sq.std() / EXPERIMENT_COUNT**0.5
        ea_mean, ea_se = ea.mean(), ea.std() / EXPERIMENT_COUNT**0.5
        sq_impr = (baseline_sqrt_pehe - sq_mean) / baseline_sqrt_pehe * 100 if baseline_sqrt_pehe != 0 else 0
        best = '  <<' if model_name == 'DESCN' else ''
        print(f'{model_name:<16s} {sq_mean:.4f} +/- {sq_se:.4f}     {sq_impr:>+6.1f}%  '
              f'{ea_mean:.4f} +/- {ea_se:.4f}{best}')
    else:
        au_mean, au_se = au.mean(), au.std() / EXPERIMENT_COUNT**0.5
        ev_mean, ev_se = ev.mean(), ev.std() / EXPERIMENT_COUNT**0.5
        au_impr = (au_mean - baseline_auuc) / baseline_auuc * 100 if baseline_auuc != 0 else 0
        ev_impr = (baseline_e_att - ev_mean) / baseline_e_att * 100 if baseline_e_att != 0 else 0
        best = '  <<' if model_name == 'DESCN' else ''
        print(f'{model_name:<16s} {au_mean:.4f} +/- {au_se:.4f}     {au_impr:>+8.1f}% '
              f'{ev_mean:.4f} +/- {ev_se:.4f}     {ev_impr:>+8.1f}%{best}')

print()
print(f'Improvement over {baseline_model_name} baseline.')
if has_ground_truth:
    print('sqrtPEHE/e_ATE: lower is better. The sqrtPEHE improvement column follows the paper: positive means lower than CFR(MMD).')
else:
    print('AUUC: higher is better. e_ATT: lower is better.')
print()

# Plot
if has_ground_truth:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    bar_colors = ['#2196F3', '#FF9800', '#4CAF50', '#F44336']
    metric_sets = [
        ('sqrtPEHE', sqrt_pehe_values),
        ('e_ATE', e_ate_values),
    ]
    for ax, (metric_name, metric_values) in zip(axes, metric_sets):
        means = [v.mean() for v in metric_values]
        errors = [v.std() / EXPERIMENT_COUNT**0.5 for v in metric_values]
        ax.bar(model_names, means, yerr=errors, color=bar_colors, capsize=5)
        ax.set_title(metric_name)
        ax.grid(axis='y', alpha=0.3)
        for i, v in enumerate(means):
            ax.text(i, v + errors[i], f'{v:.4f}', ha='center', va='bottom', fontsize=8)
    figure_path = 'model_comparison_acic.png'
else:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    bar_colors = ['#2196F3', '#FF9800', '#4CAF50', '#F44336']
    metric_sets = [
        ('AUUC', auuc_values),
        ('e_ATT', e_att_values),
    ]
    for ax, (metric_name, metric_values) in zip(axes, metric_sets):
        means = [v.mean() for v in metric_values]
        errors = [v.std() / EXPERIMENT_COUNT**0.5 for v in metric_values]
        ax.bar(model_names, means, yerr=errors, color=bar_colors, capsize=5)
        ax.set_title(f'{metric_name} - Lazada Production')
        ax.grid(axis='y', alpha=0.3)
        for i, v in enumerate(means):
            ax.text(i, v + errors[i], f'{v:.4f}', ha='center', va='bottom', fontsize=8)
    figure_path = 'model_comparison_lazada.png'

plt.tight_layout()
plt.savefig(figure_path, dpi=100, bbox_inches='tight')
plt.show()
print(f'Saved to {figure_path}')


[TARNet exp 1/20] epochs=15  batch=500  lr=0.001  l2=0.01  select=sqrt_pehe
  weights: prpsy=0 estr=0 escr=0 h1=1 h0=2 xTR=0 xCR=0 imb=0
  train: 32,000  val: 8,000  test: 40,000  treated(train): 0.385
------------------------------------------------------------


TARNet exp 1/20: 100%|██████████| 15/15 [01:51<00:00,  7.41s/ep]


------------------------------------------------------------
[TARNet exp 1/20] Best sqrt_pehe epoch=1/15: AUUC=0.0159  e_ATT=0.3147  sqrtPEHE=0.3269  e_ATE=0.2806
[TARNet exp 2/20] epochs=15  batch=500  lr=0.001  l2=0.01  select=sqrt_pehe
  weights: prpsy=0 estr=0 escr=0 h1=1 h0=2 xTR=0 xCR=0 imb=0
  train: 32,000  val: 8,000  test: 40,000  treated(train): 0.383
------------------------------------------------------------


TARNet exp 2/20: 100%|██████████| 15/15 [01:48<00:00,  7.23s/ep]


------------------------------------------------------------
[TARNet exp 2/20] Best sqrt_pehe epoch=4/15: AUUC=0.0136  e_ATT=0.2988  sqrtPEHE=0.3122  e_ATE=0.2662
[TARNet exp 3/20] epochs=15  batch=500  lr=0.001  l2=0.01  select=sqrt_pehe
  weights: prpsy=0 estr=0 escr=0 h1=1 h0=2 xTR=0 xCR=0 imb=0
  train: 32,000  val: 8,000  test: 40,000  treated(train): 0.383
------------------------------------------------------------


...

TARNet exp 8/20: 100%|██████████| 15/15 [01:48<00:00,  7.22s/ep]


------------------------------------------------------------
[TARNet exp 8/20] Best sqrt_pehe epoch=2/15: AUUC=0.0109  e_ATT=0.1284  sqrtPEHE=0.2277  e_ATE=0.1273
[TARNet exp 9/20] epochs=15  batch=500  lr=0.001  l2=0.01  select=sqrt_pehe
  weights: prpsy=0 estr=0 escr=0 h1=1 h0=2 xTR=0 xCR=0 imb=0
  train: 32,000  val: 8,000  test: 40,000  treated(train): 0.595
------------------------------------------------------------


TARNet exp 9/20: 100%|██████████| 15/15 [01:56<00:00,  7.78s/ep]


------------------------------------------------------------
[TARNet exp 9/20] Best sqrt_pehe epoch=3/15: AUUC=0.0009  e_ATT=0.1439  sqrtPEHE=0.2409  e_ATE=0.1430
[TARNet exp 10/20] epochs=15  batch=500  lr=0.001  l2=0.01  select=sqrt_pehe
  weights: prpsy=0 estr=0 escr=0 h1=1 h0=2 xTR=0 xCR=0 imb=0
  train: 32,000  val: 8,000  test: 40,000  treated(train): 0.593
------------------------------------------------------------


TARNet exp 10/20:  40%|████      | 6/15 [00:57<01:13,  8.20s/ep]

In [ ]:
# Release GPU VRAM (useful between long runs on Apple Metal / CUDA)
import gc
tf.keras.backend.clear_session()
gc.collect()
print('VRAM released.')

---


## 6. Notes

- This notebook focuses on the main neural model path: TARNet, CFR(MMD), X-network, and DESCN.
- Ablation tables for ESN variants are out of scope.
- The implementation is TensorFlow/Keras; the original source is PyTorch.
- ACIC uses the paper's Mod 4 DGP and DESCN `.npz` shape, but the original authors' CSV-to-`.npz` split manifest is not included in the local repo.

### References

- Zhong et al. (2022). *DESCN: Deep Entire Space Cross Networks for Individual Treatment Effect Estimation*. KDD '22.
- Shalit et al. (2017). *Estimating individual treatment effect: generalization bounds and algorithms*. ICML '17.
- Kunzel et al. (2019). *Metalearners for estimating heterogeneous treatment effects using machine learning*. PNAS.
- Rosenbaum and Rubin (1983). *The central role of the propensity score in observational studies for causal effects*. Biometrika.
- Original DESCN code: https://github.com/kailiang-zhong/DESCN